# 2026 MLB Pitch Data Pull: Statcast + ABS Challenges

Builds one pitch-level dataset for the 2026 MLB regular season (Opening Day 2026-03-26 through 2026-09-27): every pitch from Baseball Savant, with ABS challenge labels and derived strike-zone columns added.

**Output:** `statcast_2026_regular_abs.parquet` next to this notebook (716,534 pitches; Savant's 119 Statcast columns + 8 custom columns).

**Sources**
- **Pitch data:** Baseball Savant Statcast Search CSV export ([search page](https://baseballsavant.mlb.com/statcast_search), [field definitions](https://baseballsavant.mlb.com/csv-docs)).
- **Challenge labels:** the same export, filtered with Savant's ABS challenge filters (`hfABSFlag`) and joined back to the pitches on `game_pk`, `at_bat_number`, `pitch_number`.
- **Validation target:** Savant's [ABS challenge leaderboard](https://baseballsavant.mlb.com/leaderboard/abs-challenges).

**Notebook layout**

| Section | What it does | Hits the network? |
|---|---|---|
| 1. Setup | Imports and run settings | No |
| 2. Savant download helpers | Polite, retrying HTTP requests | No (defines functions) |
| 3. ABS challenge flags | Fetch challenge filters and join them onto pitches | No (defines functions) |
| 4. Derived columns | Zone distance, ABS call, umpire's original call | No (defines functions) |
| 5. Daily pull + save helpers | One day end to end, with data-quality counters | No (defines functions) |
| 6. Unit tests | Hand-built checks of the section 4 math | No |
| 7. Single-day smoke test *(optional)* | Pulls one date to check the pipeline | Yes, 8 requests |
| 8. Full-season pull | Pulls each day into `daily_cache/`, skipping cached days | Only for days not yet cached |
| 9. Combine, validate, save | Stitches the cache into the final file | No |
| 10. Diagnostics | Zone-column sanity checks on the full season | No |
| Appendix | Legacy pitch-only pull (no ABS columns), off by default | Yes, if enabled |

**Quick run with an existing cache:** run sections 1–6, then 9 and 10. Section 8 makes no requests when `daily_cache/` is already complete, so running it is harmless but unnecessary.

## 1. Setup: imports and run settings

Everything you might want to change is here: season dates, output file name and format, the cache folder, and how gently to hit Savant's server.

In [ ]:
import io, time
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests

# Season and output
SEASON_START, SEASON_END = "2026-03-26", "2026-09-27"  # 2026 regular season: Opening Day -> final day
OUTPUT_FORMAT = "parquet"                             # "parquet" or "csv"
OUTPUT_NAME = "statcast_2026_regular_abs"             # saved next to this notebook
CACHE_DIR = Path("daily_cache")                       # one file per finished day, so reruns resume
CHECKS_PATH = CACHE_DIR / "daily_checks.csv"          # one row of data-quality counts per pulled day
dates = [str(d) for d in pd.date_range(SEASON_START, SEASON_END).date]

# Politeness settings: requests go out one at a time, each waiting until at least REQUEST_DELAY
# seconds after the previous one finished. Rate limits / server errors back off 30s, 60s, 120s, 240s.
REQUEST_DELAY = 2
MAX_RETRIES = 4
RETRY_BACKOFF = 30
RETRY_STATUSES = {429, 500, 502, 503, 504}

## 2. Savant download helpers

- `polite_get` sends one request at a time, spaced at least `REQUEST_DELAY` seconds apart. On rate limits (429), server errors, or dropped connections it retries with exponential backoff, honoring the server's `Retry-After` header when given.
- `fetch_day` downloads one day of 2026 regular-season pitches as a DataFrame. `hfGT=R` restricts to regular-season games, so spring-training dates come back empty. The optional `extra` argument appends a filter to the query; section 3 uses it for the ABS challenge filters.

In [ ]:
BASE_URL = ("https://baseballsavant.mlb.com/statcast_search/csv?all=true&type=details"
            "&hfGT=R%7C&hfSea=2026%7C&player_type=pitcher&game_date_gt={d}&game_date_lt={d}")

session = requests.Session()
_last_request = 0.0

def polite_get(url):
    """GET with spacing between requests and exponential backoff on transient errors."""
    global _last_request
    for attempt in range(MAX_RETRIES + 1):
        time.sleep(max(0, REQUEST_DELAY - (time.time() - _last_request)))
        r, error = None, None
        try:
            r = session.get(url, timeout=120)
            if r.status_code in RETRY_STATUSES:
                error = f"HTTP {r.status_code}"
        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as e:
            error = type(e).__name__
        _last_request = time.time()
        if error is None:
            r.raise_for_status()
            return r
        if attempt == MAX_RETRIES:
            raise RuntimeError(f"Giving up after {MAX_RETRIES} retries ({error}): {url}")
        wait = RETRY_BACKOFF * 2 ** attempt
        if r is not None and r.headers.get("Retry-After", "").isdigit():
            wait = max(wait, int(r.headers["Retry-After"]))  # server told us how long to back off
        print(f"    {error} -- waiting {wait}s before retry {attempt + 1}/{MAX_RETRIES}")
        time.sleep(wait)

def fetch_day(d, extra=""):
    """Pull one day of 2026 regular-season Statcast pitch data.
    `extra` is appended to the query string (e.g. an hfABSFlag filter).
    Returns an (empty) DataFrame if there were no matching pitches that day."""
    r = polite_get(BASE_URL.format(d=d) + extra)
    if not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text))

## 3. ABS challenge flags

Statcast's regular columns don't say whether a pitch was challenged. Savant's search exposes that only as filters (`hfABSFlag`), so for each day we run the search once per filter, record which pitches came back, and join those flags onto the full pitch list by `PITCH_KEY` (`game_pk`, `at_bat_number`, `pitch_number`).

**Escaping gotcha:** the site's JavaScript escapes every `.` as `\.` before sending a filter, so the raw value `is..challengeabs..review` has to go out as `is\.\.challengeabs\.\.review|`. Without that, the server silently ignores the filter and returns every pitch. Section 5 checks for this every day.

**Columns added**

| Column | Meaning |
|---|---|
| `challengeable_pitch` | 1 if the pitch was eligible for an ABS challenge |
| `abs_flag` | 1 if the pitch was challenged |
| `challenge_overturned` | 1 if the challenge overturned the umpire's call |
| `challenger` | `batter`, `pitcher`, `catcher`, or `fielder` (the fallback when Savant flags a fielder challenge but neither pitcher nor catcher); missing if not challenged |

In [ ]:
ABS_FLAGS = {
    "abs_flag":             "is..challengeabs..review",
    "challenge_overturned": "is..challengeabs..overturned",
    "challengeable_pitch":  "is..challengeabs..opp",
    "by_batter":            "is..challengeabs..review..bat",
    "by_pitcher":           "is..challengeabs..review..pit",
    "by_catcher":           "is..challengeabs..review..cat",
    "by_fielder":           "is..challengeabs..review..fld",
}
FLAG_COLS = ["abs_flag", "challenge_overturned", "challengeable_pitch"]
BY_COLS = {"by_batter": "batter", "by_pitcher": "pitcher", "by_catcher": "catcher", "by_fielder": "fielder"}

PITCH_KEY = ["game_pk", "at_bat_number", "pitch_number"]  # unique per pitch

def abs_filter(value):
    return "&hfABSFlag=" + quote(value.replace(".", "\\.") + "|", safe=".")

def fetch_abs_flags(d):
    """One row per pitch on date d that matches any ABS filter, with a 0/1 column per filter.
    Request spacing is handled inside fetch_day."""
    parts = []
    for col, value in ABS_FLAGS.items():
        df = fetch_day(d, abs_filter(value))
        if len(df):
            parts.append(df[PITCH_KEY].assign(flag=col))
    if not parts:
        return pd.DataFrame({c: pd.Series(dtype="int64") for c in PITCH_KEY + list(ABS_FLAGS)})
    flags = (pd.concat(parts).assign(v=1)
               .pivot_table(index=PITCH_KEY, columns="flag", values="v", aggfunc="max", fill_value=0)
               .reindex(columns=list(ABS_FLAGS), fill_value=0)
               .reset_index())
    flags.columns.name = None
    return flags

def add_abs_columns(pitches, flags):
    """Join ABS flags onto the pitch data -> abs_flag, challenge_overturned, challengeable_pitch, challenger."""
    out = pitches.merge(flags, on=PITCH_KEY, how="left", validate="one_to_one")
    cols = FLAG_COLS + list(BY_COLS)
    out[cols] = out[cols].fillna(0).astype(int)
    assert (out[["by_batter", "by_pitcher", "by_catcher"]].sum(axis=1) <= 1).all(), "pitch has >1 challenger"
    # "Challenged by Fielder" = pitcher OR catcher, so it only becomes the label as a fallback
    # (flagged fielder but neither pitcher nor catcher). Unchallenged pitches stay NaN.
    out["challenger"] = pd.Series(pd.NA, index=out.index, dtype="object")
    for col, name in reversed(BY_COLS.items()):  # lowest priority first; later assignments win
        out.loc[out[col] == 1, "challenger"] = name
    out["challenger"] = pd.Categorical(out["challenger"], categories=list(BY_COLS.values()))
    return out.drop(columns=list(BY_COLS))

## 4. Derived columns: zone distance and the umpire's original call

These are computed from columns already in the data, after the season is combined (section 9). They are not stored in `daily_cache/`.

The 2026 ABS zone is modeled as 17" wide (the plate) and from `sz_bot` to `sz_top` vertically. Distance counts all four sides: wide pitches use the horizontal gap, high/low pitches the vertical gap, and pitches off a corner the diagonal to that corner. Inside the zone, the value is the distance to the nearest edge, as a negative number.

| Column | Meaning |
|---|---|
| `zone_dist_in` | Ball **center** to the zone edge, in inches. + outside, − inside |
| `miss_margin_in` | Ball **edge** to the zone (`zone_dist_in` minus the 1.45" ball radius). + ball by that much, − strike by that much |
| `abs_call` | `strike` if any part of the ball touches the zone (`miss_margin_in` ≤ 0), else `ball`; missing when location data is missing |
| `original_call` | The umpire's call before any challenge. `description` holds the **final** call, so overturned pitches are flipped. Missing for pitches that weren't a called ball/strike (swings, balls in play, automatic calls, pitchouts, HBP) |

In [ ]:
HALF_PLATE_FT  = 8.5 / 12     # half of the 17-inch zone
BALL_RADIUS_FT = 1.45 / 12    # 2.9-inch diameter baseball
ZONE_INPUTS = ["plate_x", "plate_z", "sz_top", "sz_bot"]   # all in feet; plate_x = 0 at plate center
ZONE_COLS   = ["zone_dist_in", "miss_margin_in", "abs_call"]
# Umpire location calls. Leaves out automatic_ball/strike (pitch-clock violations), pitchout, hit_by_pitch.
UMP_CALL = {"called_strike": "strike", "ball": "ball", "blocked_ball": "ball"}

def add_zone_distance(df):
    """Add how far each pitch was from the 2026 ABS zone (17" wide at mid-plate, sz_bot to sz_top).

    zone_dist_in    Ball center to the zone edge, in inches. + = outside the zone,
                    - = inside (how far in from the nearest edge).
    miss_margin_in  Ball edge to the zone, in inches (zone_dist_in minus the 1.45" radius).
                    + = ball by that much, - = strike by that much.
    abs_call        "strike" if miss_margin_in <= 0 (any part of the ball touches the zone), else "ball".

    All three are NaN/None when plate_x, plate_z, sz_top, or sz_bot is missing.
    """
    dx = df["plate_x"].abs() - HALF_PLATE_FT                                     # + = wide of the zone
    dz = np.maximum(df["sz_bot"] - df["plate_z"], df["plate_z"] - df["sz_top"])  # + = above or below it
    outside = np.hypot(np.maximum(dx, 0), np.maximum(dz, 0))  # off a corner -> diagonal distance
    inside  = np.maximum(dx, dz)                              # in the zone -> nearest edge, as a negative
    has_loc = df[ZONE_INPUTS].notna().all(axis=1)
    signed_ft = np.where(has_loc, np.where((dx > 0) | (dz > 0), outside, inside), np.nan)

    df["zone_dist_in"]   = signed_ft * 12
    df["miss_margin_in"] = df["zone_dist_in"] - BALL_RADIUS_FT * 12
    # Without has_loc, NaN <= 0 is False and rows with no location would be labeled "ball".
    df["abs_call"]       = np.where(has_loc, np.where(df["miss_margin_in"] <= 0, "strike", "ball"), None)
    return df

def add_original_call(df):
    """Add original_call: the umpire's call before any ABS challenge, "strike" or "ball".

    description holds the FINAL call, so an overturned pitch's original call is the opposite of it.
    NaN for pitches that weren't a called ball/strike (swings, balls in play, automatic_*, pitchout, HBP).
    """
    final = df["description"].map(UMP_CALL)
    overturned = df["challenge_overturned"] == 1
    assert final[overturned].notna().all(), "an overturned pitch isn't a called ball/strike"
    df["original_call"] = final.where(~overturned, final.map({"strike": "ball", "ball": "strike"}))
    return df

## 5. Daily pull and save helpers

- `pull_day(d)` does one date end to end: 1 request for all pitches, then 7 ABS filter requests (skipped on off days), then the join. It stops with an error if Savant's 25,000-row export cap was hit (truncated data) or if an ABS filter was ignored. It also returns a row of data-quality counters for `daily_checks.csv`. The four problem counters should always be 0.
- `save_output` writes the final dataset as parquet or CSV.

In [ ]:
SAVANT_ROW_CAP = 25_000  # Savant's CSV export silently truncates here

def pull_day(d):
    """All pitches on date d with the 4 ABS columns, plus a dict of data-quality checks."""
    pitches = fetch_day(d)
    if pitches.empty:  # off day -- skip the 7 ABS requests
        return pitches, {"date": d, "pitches": 0}
    if len(pitches) >= SAVANT_ROW_CAP:
        raise RuntimeError(f"{d}: hit Savant's {SAVANT_ROW_CAP:,}-row cap, so the data is truncated")
    flags = fetch_abs_flags(d)
    if (flags[list(ABS_FLAGS)].sum() >= len(pitches)).any():
        raise RuntimeError(f"{d}: an hfABSFlag filter was ignored -- check the escaping")
    day = add_abs_columns(pitches, flags)
    return day, {
        "date": d,
        "pitches": len(day),
        "challengeable": int(day["challengeable_pitch"].sum()),
        "challenged": int(day["abs_flag"].sum()),
        "overturned": int(day["challenge_overturned"].sum()),
        # Problem counters: all 0 on the 2026-04-01 test, so anything nonzero is worth a look.
        "unmatched_flags": len(flags) - len(flags.merge(pitches[PITCH_KEY], on=PITCH_KEY)),
        "overturned_not_challenged": int(((day["challenge_overturned"] == 1) & (day["abs_flag"] == 0)).sum()),
        "challenged_not_challengeable": int(((day["abs_flag"] == 1) & (day["challengeable_pitch"] == 0)).sum()),
        "challenger_mismatch": int((day["challenger"].notna() != (day["abs_flag"] == 1)).sum()),
    }

def save_output(df, name, fmt):
    if fmt == "parquet":
        path = f"{name}.parquet"
        df.to_parquet(path, index=False)
    elif fmt == "csv":
        path = f"{name}.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")  # BOM so Excel shows accented names correctly
    else:
        raise ValueError(f"OUTPUT_FORMAT must be 'parquet' or 'csv', got {fmt!r}")
    return path

## 6. Unit tests (offline)

Hand-built pitches with known answers, checking the section 4 math: an edge strike, an edge ball, dead center, a corner miss (must use the diagonal), and missing location data. Then `original_call` flipping on overturned pitches. No requests; any failure raises an error. Displays the zone test table.

In [ ]:
# Hand-built checks for add_zone_distance: sz_bot = 1.5 ft, sz_top = 3.5 ft, zone edge at |plate_x| = 0.708 ft.
zt = add_zone_distance(pd.DataFrame({
    "plate_x": [0.80, 0.85, 0.00, 0.90, np.nan],   # edge strike, edge ball, dead center, high corner, missing
    "plate_z": [2.5,  2.5,  2.5,  3.70, 2.5],
    "sz_top": 3.5, "sz_bot": 1.5,
}))
corner = np.hypot((0.90 - HALF_PLATE_FT) * 12, (3.70 - 3.5) * 12)  # 2.3" wide, 2.4" high -> 3.32" diagonal

assert np.isclose(zt.loc[0, "miss_margin_in"], -0.35) and zt.loc[0, "abs_call"] == "strike"
assert np.isclose(zt.loc[1, "miss_margin_in"],  0.25) and zt.loc[1, "abs_call"] == "ball"
assert zt.loc[2, "miss_margin_in"] < 0 and zt.loc[2, "abs_call"] == "strike"
assert np.isclose(zt.loc[3, "zone_dist_in"], corner) and zt.loc[3, "abs_call"] == "ball"  # diagonal, not max(dx, dz)
assert zt.loc[4, ZONE_COLS].isna().all()                                                   # missing stays missing

# Hand-built checks for add_original_call: overturned pitches flip, everything else keeps its called result.
oc = add_original_call(pd.DataFrame({
    "description": ["called_strike", "ball", "called_strike", "ball", "blocked_ball", "swinging_strike"],
    "challenge_overturned": [0, 0, 1, 1, 1, 0],
}))
assert oc["original_call"].iloc[:5].tolist() == ["strike", "ball", "ball", "strike", "strike"]
assert pd.isna(oc.loc[5, "original_call"])                                                 # swing -> no called result
zt

## 7. Optional: single-day smoke test (8 requests)

Pulls one date to confirm the pipeline works before (or after) a full run: first the raw pitches, then the ABS flags with consistency checks (filters not ignored, join didn't add or drop rows, every flagged pitch matched, overturned ⇒ challenged ⇒ challengeable, challenger set only on challenged pitches). Skip this when you already have a complete cache.

`TEST_DATE` must be on or after Opening Day (2026-03-26). Earlier dates are spring training and return zero rows.

In [ ]:
TEST_DATE = "2026-04-01"
test_df = fetch_day(TEST_DATE)
print(f"{TEST_DATE}: {len(test_df)} rows")
test_df.head()

In [ ]:
flags = fetch_abs_flags(TEST_DATE)
test_abs = add_abs_columns(test_df, flags)

assert (flags[list(ABS_FLAGS)].sum() < len(test_df)).all(), "an hfABSFlag filter was ignored -- check the escaping"
assert len(test_abs) == len(test_df)                                          # join didn't add/drop rows
assert len(flags.merge(test_df[PITCH_KEY], on=PITCH_KEY)) == len(flags)       # every flagged pitch matched
assert (test_abs["challenge_overturned"] <= test_abs["abs_flag"]).all()       # overturned => challenged
assert (test_abs["abs_flag"] <= test_abs["challengeable_pitch"]).all()        # challenged => challengeable
assert (test_abs["challenger"].notna() == (test_abs["abs_flag"] == 1)).all()  # challenger set iff challenged

print(f"{TEST_DATE}: {len(test_abs)} pitches | {test_abs['challengeable_pitch'].sum()} challengeable | "
      f"{test_abs['abs_flag'].sum()} challenged | {test_abs['challenge_overturned'].sum()} overturned")
pd.crosstab(test_abs["challenger"], test_abs["challenge_overturned"], margins=True)

## 8. Full-season pull (cached and resumable)

Pulls every date from `SEASON_START` to `SEASON_END` with `pull_day` and saves each finished day to `daily_cache/<date>.parquet`, plus a row in `daily_cache/daily_checks.csv`.

- **Resumable:** days already listed in `daily_checks.csv` are skipped without any request. If the run is interrupted, just re-run this cell.
- **Server load:** a game day takes 8 requests and an off day 1, strictly one at a time and at least 2s apart. A fresh pull is about 1,470 requests, roughly **1 to 1.5 hours**.
- **Fresh pull:** cached days are never refreshed. Delete `daily_cache/` to re-download everything (for example, if Savant has corrected data).

In [ ]:
CACHE_DIR.mkdir(exist_ok=True)
checks = pd.read_csv(CHECKS_PATH).to_dict("records") if CHECKS_PATH.exists() else []
done = {c["date"] for c in checks}

print(f"{len(done)} of {len(dates)} days already cached")
for i, d in enumerate(dates, 1):
    if d in done:
        continue
    day, chk = pull_day(d)
    if not day.empty:
        day.to_parquet(CACHE_DIR / f"{d}.parquet", index=False)
    checks.append(chk)
    pd.DataFrame(checks).to_csv(CHECKS_PATH, index=False)  # written after the day file, so a crash just re-pulls that day
    print(f"[{i}/{len(dates)}] {d}: {chk['pitches']:,} pitches, {chk.get('challenged', 0)} challenged")

## 9. Combine, validate, and save

Reads the cache from disk (so it works even if section 8 wasn't run this session) and:
1. Confirms every season date has been pulled.
2. Stacks the cached days into `season` and checks that no pitches were lost or duplicated.
3. Adds the section 4 columns (`zone_dist_in`, `miss_margin_in`, `abs_call`, `original_call`).
4. Reports off days and any day with a nonzero data-quality counter.
5. Saves `statcast_2026_regular_abs.parquet` and shows challenges by challenger and outcome.

In [ ]:
checks_df = pd.read_csv(CHECKS_PATH)
checks_df = checks_df[checks_df["date"].isin(dates)]
missing = sorted(set(dates) - set(checks_df["date"]))
assert not missing, f"{len(missing)} days not pulled yet (e.g. {missing[:3]}) -- run section 8"

season = pd.concat([pd.read_parquet(CACHE_DIR / f"{d}.parquet") for d in dates
                    if (CACHE_DIR / f"{d}.parquet").exists()], ignore_index=True)
season["challenger"] = pd.Categorical(season["challenger"], categories=list(BY_COLS.values()))

assert len(season) == checks_df["pitches"].sum()                 # every cached day made it in
assert not season.duplicated(PITCH_KEY).any(), "duplicate pitches across days"
season = add_zone_distance(season)  # adds zone_dist_in, miss_margin_in, abs_call
season = add_original_call(season)  # adds original_call (umpire's call before any challenge)

problem_cols = ["unmatched_flags", "overturned_not_challenged", "challenged_not_challengeable", "challenger_mismatch"]
problems = checks_df[checks_df.reindex(columns=problem_cols).fillna(0).sum(axis=1) > 0]
print("Days with 0 pitches:", checks_df.loc[checks_df["pitches"] == 0, "date"].tolist())
print("Days with data-quality flags:", "none" if problems.empty else f"\n{problems.to_string(index=False)}")

path = save_output(season, OUTPUT_NAME, OUTPUT_FORMAT)
print(f"Saved {len(season):,} pitches x {season.shape[1]} columns to {path}")
print(f"{season['challengeable_pitch'].sum():,} challengeable | {season['abs_flag'].sum():,} challenged | "
      f"{season['challenge_overturned'].sum():,} overturned")
pd.crosstab(season["challenger"], season["challenge_overturned"], margins=True)

## 10. Diagnostics: zone columns on the full season

- How many rows are missing the zone columns (no tracked location).
- The distribution of `miss_margin_in`.
- Every `description` value, to see which pitches count as called balls/strikes.
- Agreement between `abs_call` and the umpire on called pitches:
  - **vs. `description` (final call):** challenged pitches here already carry the ABS ruling.
  - **Unchallenged only:** the umpire's call stood.
  - **vs. `original_call`:** the umpire's call before any challenge, the closest measure of umpire accuracy against our zone.

In [ ]:
print("Rows missing zone columns:\n" + season[ZONE_COLS].isna().sum().to_string())
print("\nmiss_margin_in (inches):\n" + season["miss_margin_in"].describe().round(2).to_string())
print("\ndescription values:\n" + season["description"].value_counts().to_string())

called = season[season["description"].isin(UMP_CALL) & season["abs_call"].notna()]
agree = called["abs_call"] == called["description"].map(UMP_CALL)
print(f"\nCalled pitches: {len(called):,} | abs_call agrees with description: {agree.mean():.2%}")
print(f"Unchallenged called pitches only: {agree[called['abs_flag'] == 0].mean():.2%}")
print(f"Original umpire call (before challenges) agrees with abs_call: {(called['original_call'] == called['abs_call']).mean():.2%}")
pd.crosstab(called["original_call"], called["abs_call"], margins=True)

## Appendix: legacy pitch-only pull (off by default)

The project's first full-season pull. It downloads raw pitches only, **without** the ABS or custom columns, to `statcast_2026_regular.parquet`. Sections 8–9 replace it, so it is disabled. Set `RUN_LEGACY_PULL = True` to run it (about 190 requests).

In [ ]:
RUN_LEGACY_PULL = False

if RUN_LEGACY_PULL:
    frames = []
    for d in pd.date_range("2026-03-26", "2026-10-01").date:
        df = fetch_day(d)
        if not df.empty:
            frames.append(df)
        print(d, len(df), "rows")
        time.sleep(1)  # be polite to the server

    statcast_2026 = pd.concat(frames, ignore_index=True)
    statcast_2026.to_parquet("statcast_2026_regular.parquet")
    print(statcast_2026.shape)